# Initialize a Declarative Automation Bundle, promote the same Lakeflow Job from dev to prod, and prove the target variables substituted

The platform lead was already moving the team off click-deploy and last week's prod overwrite incident (the prod job shipped the dev catalog name and overwrote three days of analytics) is the final push. The auditor wants a one-command deploy path with target variables proving dev and prod cannot share state. You have one session to bundle-init, define the catalog variable, deploy both targets, prove the prod job reads the prod catalog, and write a short runbook the on-call engineer can use without you next time.

The hands-on work:

- Pre-create two UC schemas (`labrun_bundle_dev`, `labrun_bundle_prod`) that the deployed jobs will reference.
- Author a `databricks.yml` bundle file with one job, two targets (dev, prod), and a `catalog_schema` variable that differs per target.
- Deploy the bundle locally using the Databricks CLI v0.220+ (`databricks bundle deploy --target dev` then `--target prod`).
- Confirm via the workspace API that the dev job points at `workspace.default.labrun_bundle_dev` and the prod job points at `workspace.default.labrun_bundle_prod`.
- Write the bundle.yml content into a notebook variable `bundle_yaml_text` so the validator can parse it and assert on the variable structure.

**Time.** Plan for about 70 minutes hands-on. Most of it is reading the bundle CLI output and chasing typos in YAML. Budget up to 120 minutes for the session window.

**Cost.** Zero dollars. Bundle deploys are metadata-only.

This lab maps to Databricks DE Associate Domain 5 (Implementing CI/CD, ~10% provisional).

**Rename callout.** If your other prep material says "Databricks Asset Bundles" or "DABs" it means Declarative Automation Bundles on the current exam. The CLI subcommand is still `databricks bundle ...` and the file is still `databricks.yml`. The variable surface keeps the older "Automation Bundle variables" name.

**Local prerequisite.** This lab requires the Databricks CLI v0.220 or later installed locally on your machine AND authenticated against your Free Edition workspace via `databricks auth login` or a `~/.databrickscfg` profile. Run the `databricks bundle ...` commands in your local shell; the notebook validates the deployed jobs via the workspace API.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and the Databricks SDK. Pinned versions per
# LAB-CREATION-BLUEPRINT.md build rules.

!pip install --quiet labrun-checks==0.6.0 databricks-sdk==0.40.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. The bundle YAML lives on the student's
# local machine; the notebook holds a copy of the text in bundle_yaml_text
# so the validator can parse it.

import atexit
import getpass
import json
import sys
import time

import yaml  # available in Databricks runtimes; pip install pyyaml locally if needed

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupResource,
)

LAB_ID = "declarative-automation-bundles-with-cli"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID

CATALOG = "workspace"
PARENT_SCHEMA = "default"
DEV_SCHEMA = "labrun_bundle_dev"
PROD_SCHEMA = "labrun_bundle_prod"
DEV_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{DEV_SCHEMA}"
PROD_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{PROD_SCHEMA}"

DEV_LOG_TABLE = "deploy_log"
PROD_LOG_TABLE = "deploy_log"
DEV_LOG_FQN = f"{DEV_SCHEMA_FQN}.{DEV_LOG_TABLE}"
PROD_LOG_FQN = f"{PROD_SCHEMA_FQN}.{PROD_LOG_TABLE}"

JOB_NAME = "labrun-bundle-job"

# Set by the student in Task 1 (the deployed-job's name prefix is
# `[dev <user>]` for dev and bare for prod per Databricks bundle convention).
cli_profile_name = None
dev_job_id = None
prod_job_id = None

# Filled in Task 2 by the student copy-pasting their bundle.yml content.
bundle_yaml_text = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials, resolve the
# Starter SQL warehouse, expose run_sql().

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL: ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired:")
    print(f"  {exc}")
    raise SystemExit(1)
CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found. Recreate the Starter warehouse and re-run.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid, statement=statement, wait_timeout="50s",
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(f"SQL did not finish in {wait_seconds}s. Last state: {state}.")
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)
    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan. Both deployed jobs
# are critical because running deploys consume daily quota. The local
# bundle directory and ~/.databrickscfg profile are outside the workspace
# and are NOT in the manifest; the cleanup cell prints a reminder.


def _find_jobs_by_name_contains(needle):
    found = []
    try:
        for job in w.jobs.list():
            if job.settings and (job.settings.name or "").endswith(needle) or (job.settings and needle in (job.settings.name or "")):
                found.append(job)
    except Exception:
        pass
    return found


CLEANUP_MANIFEST = [
    CleanupResource(
        type="lakeflow_job",
        id=f"prod:{JOB_NAME}",
        region="databricks",
        critical=True,
        cli_delete_command="cd labrun-bundle && databricks bundle destroy --target prod --auto-approve",
    ),
    CleanupResource(
        type="lakeflow_job",
        id=f"dev:{JOB_NAME}",
        region="databricks",
        critical=True,
        cli_delete_command="cd labrun-bundle && databricks bundle destroy --target dev --auto-approve",
    ),
    CleanupResource(
        type="workspace_path",
        id=f"/Workspace/Users/{{CALLER_USER}}/.bundle/labrun-bundle/prod",
        region="databricks",
        cli_delete_command="databricks workspace delete --recursive <prod bundle artifacts path>",
    ),
    CleanupResource(
        type="workspace_path",
        id=f"/Workspace/Users/{{CALLER_USER}}/.bundle/labrun-bundle/dev",
        region="databricks",
        cli_delete_command="databricks workspace delete --recursive <dev bundle artifacts path>",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=PROD_LOG_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {PROD_LOG_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=DEV_LOG_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {DEV_LOG_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=PROD_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {PROD_SCHEMA_FQN} CASCADE\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=DEV_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {DEV_SCHEMA_FQN} CASCADE\"",
    ),
]
# Patch workspace_path ids now that CALLER_USER is known.
for res in CLEANUP_MANIFEST:
    if res.type == "workspace_path":
        res.id = res.id.replace("{CALLER_USER}", CALLER_USER)
        res.cli_delete_command = f"databricks workspace delete --recursive {res.id}"


class DatabricksCleanupAdapter:
    def delete_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        elif rtype == "workspace_path":
            try:
                w.workspace.delete(rid, recursive=True)
            except Exception:
                pass
        elif rtype == "lakeflow_job":
            # rid is "dev:<job_name>" or "prod:<job_name>".
            target, name = rid.split(":", 1)
            # Match by name. Dev jobs are prefixed `[dev <user>]` by the
            # bundle CLI; prod jobs are bare.
            for job in w.jobs.list():
                if not job.settings or not job.settings.name:
                    continue
                if target == "prod" and job.settings.name == name:
                    self._delete_job(job.job_id)
                if target == "dev" and name in (job.settings.name or "") and "[dev" in (job.settings.name or ""):
                    self._delete_job(job.job_id)
        else:
            raise ValueError(f"Unknown resource type {rtype!r}")

    def _delete_job(self, job_id):
        try:
            runs = w.jobs.list_runs(job_id=job_id, active_only=True)
            for run in (runs.runs or []):
                try:
                    w.jobs.cancel_run(run.run_id)
                except Exception:
                    pass
        except Exception:
            pass
        try:
            w.jobs.delete(job_id)
        except Exception:
            pass

    def describe_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "workspace_path":
            try:
                w.workspace.get_status(rid)
                return True
            except Exception:
                return False
        if rtype == "lakeflow_job":
            target, name = rid.split(":", 1)
            for job in w.jobs.list():
                if not job.settings or not job.settings.name:
                    continue
                if target == "prod" and job.settings.name == name:
                    return True
                if target == "dev" and name in (job.settings.name or "") and "[dev" in (job.settings.name or ""):
                    return True
            return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        try:
            for job in w.jobs.list():
                tags = (job.settings.tags if job.settings else None) or {}
                if tags.get(LAB_TAG_KEY) == lab_slug:
                    arns.append(f"job:{job.settings.name}")
        except Exception:
            pass
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


orphans = CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")
if orphans:
    print(f"BLOCKED: existing objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Pre-create the dev and prod schemas, confirm CLI is authenticated

The bundle assumes both target schemas already exist (production-style bundles let the bundle itself declare schemas, but the Lab keeps the surface focused on the variable-substitution lesson). Build the two schemas, then record the CLI profile name you used for `databricks bundle deploy`.

In a separate terminal on your local machine, run:

```bash
databricks auth login --host <your-workspace-url> --profile labrun-free-edition
```

This adds a profile to `~/.databrickscfg`. Record `labrun-free-edition` as `cli_profile_name` in the notebook below so the validator can confirm the value is a non-empty string. (The validator cannot inspect your local CLI config; it only checks that the notebook variable is set, and that a workspace identity matching your signed-in user exists.)

In [ ]:
# NBVAL_SKIP
# Task 1: Pre-create both schemas, set cli_profile_name, pre-create the
# two deploy_log tables (the bundle deploys reference them).

# YOUR CODE: Run CREATE SCHEMA IF NOT EXISTS DEV_SCHEMA_FQN.
# YOUR CODE: Tag dev schema with ALTER SCHEMA DEV_SCHEMA_FQN SET TAGS.
# YOUR CODE: Run CREATE SCHEMA IF NOT EXISTS PROD_SCHEMA_FQN.
# YOUR CODE: Tag prod schema.

# Pre-create the two deploy_log tables so the deployed jobs have a target.
for fqn in (DEV_LOG_FQN, PROD_LOG_FQN):
    # YOUR CODE: Run CREATE TABLE IF NOT EXISTS fqn (
    #   deployed_at STRING, target STRING, catalog_schema STRING
    # ) USING DELTA via run_sql().
    # YOUR CODE: Tag with ALTER TABLE fqn SET TAGS.
    pass

# YOUR CODE: Assign cli_profile_name to the string name of the CLI profile
# you configured (default: "labrun-free-edition"). This is informational;
# the validator only checks the variable is a non-empty string.

global cli_profile_name
# YOUR CODE: cli_profile_name = "labrun-free-edition"

print(f"Dev schema:  {DEV_SCHEMA_FQN}")
print(f"Prod schema: {PROD_SCHEMA_FQN}")
print(f"CLI profile recorded: {cli_profile_name}")
print()
print("In a local terminal, run:")
print(f"  databricks auth login --host <your-workspace-url> --profile {cli_profile_name}")
print("then complete the browser OAuth flow.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Both schemas exist and are tagged. Both deploy_log tables
# exist. cli_profile_name is a non-empty string. SCIM Me returns the user.


def checkpoint_1(session):
    try:
        for schema_fqn, schema_label in [
            (DEV_SCHEMA_FQN, "dev"),
            (PROD_SCHEMA_FQN, "prod"),
        ]:
            catalog, parent, schema = schema_fqn.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            if not run_sql(sql)["rows"]:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Schema {schema_fqn} ({schema_label}) not found.",
                )
            tag_sql = (
                "SELECT tag_value FROM system.information_schema.schema_tags "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}' "
                f"AND tag_name = '{LAB_TAG_KEY}'"
            )
            if not any(r[0] == LAB_TAG_VALUE for r in run_sql(tag_sql)["rows"]):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Schema {schema_fqn} missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}."
                    ),
                )

        for fqn in (DEV_LOG_FQN, PROD_LOG_FQN):
            catalog, parent, table = fqn.split(".")[0], fqn.split(".")[1], fqn.split(".")[2]
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{parent}' "
                f"AND table_name = '{table}'"
            )
            if not run_sql(sql)["rows"]:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Table {fqn} not found. Did CREATE TABLE run?",
                )

        if not cli_profile_name or not isinstance(cli_profile_name, str):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "cli_profile_name is not set. Assign a string naming the CLI "
                    "profile you configured (e.g., 'labrun-free-edition')."
                ),
            )

        try:
            me_now = w.current_user.me()
            if not me_now or not (me_now.user_name or me_now.emails):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        "Workspace identity check (SCIM Me) returned no user. The "
                        "session credentials may be invalid."
                    ),
                )
        except Exception as exc:
            return CheckpointResult(
                status="fail",
                error_reason=f"SCIM Me call failed: {exc}",
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is missing: a schema row, a schema tag, a deploy_log table, or `cli_profile_name`. Each is a different SQL or Python assignment.

</details>

<details><summary>Hint 2 (direction)</summary>

Four SQL CREATE/ALTER pairs (two schemas, two tables) and one Python assignment. The schemas are pre-created so the deployed jobs have a UC location. The `cli_profile_name` is informational; just assign the string you used on the CLI side.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {DEV_SCHEMA_FQN}")
run_sql(f"ALTER SCHEMA {DEV_SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")
run_sql(f"CREATE SCHEMA IF NOT EXISTS {PROD_SCHEMA_FQN}")
run_sql(f"ALTER SCHEMA {PROD_SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

for fqn in (DEV_LOG_FQN, PROD_LOG_FQN):
    run_sql(
        f"CREATE TABLE IF NOT EXISTS {fqn} ("
        f"  deployed_at STRING, target STRING, catalog_schema STRING"
        f") USING DELTA"
    )
    run_sql(f"ALTER TABLE {fqn} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

global cli_profile_name
cli_profile_name = "labrun-free-edition"
```

</details>

**Wallet check.** Still at $0.00. Two empty schemas and two empty tables. The CLI side has run no compute.

## Task 2: Author `databricks.yml` and paste its content into `bundle_yaml_text`

On your local machine, create a folder `labrun-bundle/` and put the following in `labrun-bundle/databricks.yml`:

```yaml
bundle:
  name: labrun-bundle

variables:
  catalog_schema:
    description: "Three-level UC location the bundle job writes to"
    default: workspace.default.labrun_bundle_dev

resources:
  jobs:
    labrun-bundle-job:
      name: labrun-bundle-job
      tags:
        labrun_lab_slug: declarative-automation-bundles-with-cli
      tasks:
        - task_key: log_deploy
          notebook_task:
            notebook_path: ./log_deploy.py
            base_parameters:
              catalog_schema: ${var.catalog_schema}

targets:
  dev:
    mode: development
    default: true
    workspace:
      host: https://YOUR-WORKSPACE-HOST
    variables:
      catalog_schema: workspace.default.labrun_bundle_dev
    resources:
      jobs:
        labrun-bundle-job:
          tags:
            bundle_target: dev
  prod:
    mode: production
    workspace:
      host: https://YOUR-WORKSPACE-HOST
    variables:
      catalog_schema: workspace.default.labrun_bundle_prod
    resources:
      jobs:
        labrun-bundle-job:
          tags:
            bundle_target: prod
```

Also create `labrun-bundle/log_deploy.py` with a single line that uses the parameter:

```python
# Databricks notebook source
dbutils.widgets.text("catalog_schema", "")
catalog_schema = dbutils.widgets.get("catalog_schema")
print(f"deploy logged for catalog_schema={catalog_schema}")
```

Then paste the entire `databricks.yml` content into the `bundle_yaml_text` notebook variable below so the validator can parse it.

In [ ]:
# NBVAL_SKIP
# Task 2: Copy your local databricks.yml content into bundle_yaml_text.

global bundle_yaml_text

# YOUR CODE: Set bundle_yaml_text to a multiline string containing the
# EXACT contents of your labrun-bundle/databricks.yml file. The validator
# parses it via yaml.safe_load and walks the structure to confirm:
#  - resources.jobs.labrun-bundle-job exists
#  - targets.dev and targets.prod both exist
#  - variables.catalog_schema has a default value
#  - targets.prod.variables.catalog_schema overrides the default
#  - the dev override resolves to workspace.default.labrun_bundle_dev
#  - the prod override resolves to workspace.default.labrun_bundle_prod

# Example (paste your actual file):
# bundle_yaml_text = """
# bundle:
#   name: labrun-bundle
# ...
# """

print(f"bundle_yaml_text length: {len(bundle_yaml_text or '')} chars")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Parse bundle_yaml_text and assert the structural shape:
# one job, two targets, a per-target catalog_schema override resolving to
# the dev and prod schemas.


def checkpoint_2(session):
    try:
        if not bundle_yaml_text or not bundle_yaml_text.strip():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "bundle_yaml_text is empty. Paste the full contents of your "
                    "labrun-bundle/databricks.yml file into the notebook variable."
                ),
            )
        try:
            doc = yaml.safe_load(bundle_yaml_text)
        except yaml.YAMLError as exc:
            return CheckpointResult(
                status="fail",
                error_reason=f"bundle_yaml_text is not valid YAML: {exc}",
            )
        if not isinstance(doc, dict):
            return CheckpointResult(
                status="fail",
                error_reason=f"Top-level YAML is not a mapping; got {type(doc).__name__}.",
            )

        resources = doc.get("resources") or {}
        jobs = resources.get("jobs") or {}
        if JOB_NAME not in jobs:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"resources.jobs is missing the {JOB_NAME!r} entry. "
                    f"Got jobs: {list(jobs.keys())}."
                ),
            )

        targets = doc.get("targets") or {}
        missing_targets = {"dev", "prod"} - set(targets.keys())
        if missing_targets:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"targets missing: {sorted(missing_targets)}. "
                    f"Define both targets.dev and targets.prod."
                ),
            )

        variables = doc.get("variables") or {}
        catalog_var = variables.get("catalog_schema")
        if not isinstance(catalog_var, dict) or "default" not in catalog_var:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "variables.catalog_schema must be a mapping with a 'default' key. "
                    "Define it at the top level of databricks.yml."
                ),
            )

        dev_vars = ((targets.get("dev") or {}).get("variables") or {})
        prod_vars = ((targets.get("prod") or {}).get("variables") or {})
        dev_resolved = dev_vars.get("catalog_schema") or catalog_var["default"]
        prod_resolved = prod_vars.get("catalog_schema")
        if not prod_resolved:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "targets.prod.variables.catalog_schema is not set. The prod "
                    "target must override the default."
                ),
            )
        if dev_resolved != DEV_SCHEMA_FQN:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"dev catalog_schema resolves to {dev_resolved!r}; expected "
                    f"{DEV_SCHEMA_FQN!r}."
                ),
            )
        if prod_resolved != PROD_SCHEMA_FQN:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"prod catalog_schema resolves to {prod_resolved!r}; expected "
                    f"{PROD_SCHEMA_FQN!r}."
                ),
            )

        # Verify the job task references ${var.catalog_schema}.
        job_def = jobs[JOB_NAME]
        tasks = job_def.get("tasks") or []
        if not tasks:
            return CheckpointResult(
                status="fail",
                error_reason=f"Job {JOB_NAME!r} has no tasks.",
            )
        blob = json.dumps(job_def)
        if "${var.catalog_schema}" not in blob:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Job task config does not reference ${var.catalog_schema}. "
                    "Pass it as a base_parameter or task parameter so the variable "
                    "actually substitutes at deploy time."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is missing: the job entry, a target, the variable default, the prod override, the resolved value, or the task's `${var.catalog_schema}` reference. Each is a YAML edit.

</details>

<details><summary>Hint 2 (direction)</summary>

Paste your full `databricks.yml` text into `bundle_yaml_text` as a Python triple-quoted string. Make sure the prod target's `variables.catalog_schema` is `workspace.default.labrun_bundle_prod` exactly, and that one of the job's tasks has `base_parameters: catalog_schema: ${var.catalog_schema}` so the variable actually substitutes.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
global bundle_yaml_text
bundle_yaml_text = """
bundle:
  name: labrun-bundle

variables:
  catalog_schema:
    description: Three-level UC location the bundle job writes to
    default: workspace.default.labrun_bundle_dev

resources:
  jobs:
    labrun-bundle-job:
      name: labrun-bundle-job
      tags:
        labrun_lab_slug: declarative-automation-bundles-with-cli
      tasks:
        - task_key: log_deploy
          notebook_task:
            notebook_path: ./log_deploy.py
            base_parameters:
              catalog_schema: ${var.catalog_schema}

targets:
  dev:
    mode: development
    default: true
    variables:
      catalog_schema: workspace.default.labrun_bundle_dev
    resources:
      jobs:
        labrun-bundle-job:
          tags:
            bundle_target: dev
  prod:
    mode: production
    variables:
      catalog_schema: workspace.default.labrun_bundle_prod
    resources:
      jobs:
        labrun-bundle-job:
          tags:
            bundle_target: prod
"""
```

</details>

**Wallet check.** Still at $0.00. YAML parsing happens locally in the notebook kernel; nothing on the warehouse.

## Task 3: Deploy the dev target from your local CLI and confirm in the workspace

In your local terminal, with the working directory set to `labrun-bundle/`:

```bash
databricks bundle validate --target dev --profile labrun-free-edition
databricks bundle deploy --target dev --profile labrun-free-edition
```

Validate must succeed (no errors) before deploy will run. After deploy, the workspace gets a new Lakeflow Job whose name follows the dev convention `[dev <your-user>] labrun-bundle-job` (the bundle CLI prefixes development-mode deploys with `[dev <user>]`). Capture the deployed job's job_id in the notebook by listing jobs that match the dev-naming pattern.

In [ ]:
# NBVAL_SKIP
# Task 3: Find the deployed dev job by name and capture its job_id.

global dev_job_id

# YOUR CODE: List jobs via w.jobs.list() and find the one whose name
# contains both "[dev " and "labrun-bundle-job". Assign its job_id to
# dev_job_id.

if dev_job_id is None:
    print("dev_job_id is None.")
    print("Run `databricks bundle deploy --target dev --profile labrun-free-edition`")
    print("from your local shell with cwd = labrun-bundle/, then re-run this cell.")
else:
    info = w.jobs.get(dev_job_id)
    print(f"Dev job found: id={dev_job_id} name={info.settings.name!r}")
    print(f"Tags: {info.settings.tags}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: dev_job_id resolves to a job whose task config references
# the dev catalog schema and whose tags include bundle_target=dev.


def checkpoint_3(session):
    try:
        if dev_job_id is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "dev_job_id is None. Run `databricks bundle deploy --target dev` "
                    "from your local shell, then list jobs and assign the dev job's "
                    "job_id."
                ),
            )
        info = w.jobs.get(dev_job_id)
        if not info or not info.settings:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not fetch job {dev_job_id}.",
            )
        name = info.settings.name or ""
        if "labrun-bundle-job" not in name:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Job name {name!r} does not contain 'labrun-bundle-job'. "
                    f"Did you capture the wrong job?"
                ),
            )
        if "[dev " not in name:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Job name {name!r} does not have the bundle-dev '[dev <user>]' "
                    f"prefix. Confirm you deployed with --target dev."
                ),
            )
        tags = info.settings.tags or {}
        if tags.get("bundle_target") != "dev":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Dev job tags missing bundle_target=dev. Tags: {tags}."
                ),
            )
        if tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Dev job is missing the labrun lab tag. Tags: {tags}."
                ),
            )
        blob = json.dumps([t.as_dict() if hasattr(t, "as_dict") else None for t in (info.settings.tasks or [])])
        if DEV_SCHEMA_FQN not in blob:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Dev job task config does not reference {DEV_SCHEMA_FQN}. The "
                    f"variable substitution may have skipped the dev override."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is wrong: dev_job_id not set, job name wrong, tags missing, or the task config does not reference the dev schema. Each shape has a different fix.

</details>

<details><summary>Hint 2 (direction)</summary>

Use `w.jobs.list()` and scan for a job whose `name` contains both `"[dev "` and `"labrun-bundle-job"`. The bundle CLI's development mode auto-prefixes job names. Assign the matched job's `job_id` to `dev_job_id` (declare `global dev_job_id`).

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
global dev_job_id
for job in w.jobs.list():
    name = (job.settings.name or "") if job.settings else ""
    if "[dev " in name and "labrun-bundle-job" in name:
        dev_job_id = job.job_id
        break
```

If the dev job is missing entirely, the bundle deploy did not run; check your local terminal for errors and re-run `databricks bundle deploy --target dev --profile labrun-free-edition`.

</details>

**Wallet check.** Still at $0.00. Bundle deploys are metadata operations. The deployed dev job has not run yet; that is intentional. The lab proves variable substitution, not job execution.

## Task 4: Deploy the prod target and prove dev and prod point at different catalogs

In your local terminal:

```bash
databricks bundle validate --target prod --profile labrun-free-edition
databricks bundle deploy --target prod --profile labrun-free-edition
```

The prod job is named `labrun-bundle-job` (no `[dev <user>]` prefix in production mode). Capture its `job_id` as `prod_job_id` and confirm:

- The prod job's task config references `workspace.default.labrun_bundle_prod` (the prod variable override resolved).
- The prod job's tags include `bundle_target=prod`.
- `prod_job_id != dev_job_id` (two distinct Lakeflow Jobs).

After Task 4, you have two Lakeflow Jobs that share one bundle source but point at different UC catalogs. Variable substitution proves dev and prod cannot accidentally share state.

In [ ]:
# NBVAL_SKIP
# Task 4: Find the deployed prod job by name and capture its job_id.

global prod_job_id

# YOUR CODE: List jobs via w.jobs.list() and find the one whose name is
# EXACTLY "labrun-bundle-job" (no [dev <user>] prefix). Assign its job_id
# to prod_job_id.

if prod_job_id is None:
    print("prod_job_id is None.")
    print("Run `databricks bundle deploy --target prod --profile labrun-free-edition`")
    print("from your local shell, then re-run this cell.")
else:
    info = w.jobs.get(prod_job_id)
    print(f"Prod job found: id={prod_job_id} name={info.settings.name!r}")
    print(f"Tags: {info.settings.tags}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: prod_job_id resolves to a separate job whose task config
# references the prod catalog schema and whose tags include
# bundle_target=prod. dev_job_id and prod_job_id are different.


def checkpoint_4(session):
    try:
        if prod_job_id is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "prod_job_id is None. Run `databricks bundle deploy --target prod` "
                    "and capture the deployed job's id."
                ),
            )
        if dev_job_id is None:
            return CheckpointResult(
                status="fail",
                error_reason="dev_job_id is None. Complete Task 3 first.",
            )
        if prod_job_id == dev_job_id:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"prod_job_id ({prod_job_id}) equals dev_job_id. Dev and prod "
                    f"must be two distinct Lakeflow Jobs."
                ),
            )
        info = w.jobs.get(prod_job_id)
        if not info or not info.settings:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not fetch job {prod_job_id}.",
            )
        name = info.settings.name or ""
        if name != "labrun-bundle-job":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Prod job name is {name!r}; expected 'labrun-bundle-job' "
                    f"exactly (no '[dev <user>]' prefix)."
                ),
            )
        tags = info.settings.tags or {}
        if tags.get("bundle_target") != "prod":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Prod job tags missing bundle_target=prod. Tags: {tags}."
                ),
            )
        if tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Prod job missing labrun lab tag. Tags: {tags}."
                ),
            )
        blob = json.dumps([t.as_dict() if hasattr(t, "as_dict") else None for t in (info.settings.tasks or [])])
        if PROD_SCHEMA_FQN not in blob:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Prod job task config does not reference {PROD_SCHEMA_FQN}. "
                    f"The prod variable override did not resolve correctly."
                ),
            )
        if DEV_SCHEMA_FQN in blob:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Prod job task config references {DEV_SCHEMA_FQN}. Did you pass "
                    f"a --var override on the prod deploy? Remove it; let the bundle "
                    f"YAML drive the substitution."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names exactly what is wrong: prod_job_id not set, IDs are equal, prod job referenced the dev schema, or the tag is missing. Each one is a separate fix.

</details>

<details><summary>Hint 2 (direction)</summary>

The prod job's name is exactly `labrun-bundle-job` (no prefix). Scan `w.jobs.list()` for that exact name. If you also see a job named `labrun-bundle-job` with the dev prefix, that is the dev job; the prod one has no brackets.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
global prod_job_id
for job in w.jobs.list():
    name = (job.settings.name or "") if job.settings else ""
    if name == "labrun-bundle-job":
        prod_job_id = job.job_id
        break
```

If the prod job's task config shows the dev schema, you ran `databricks bundle deploy --target prod --var "catalog_schema=workspace.default.labrun_bundle_dev"` by mistake. CLI flags override the YAML; remove the flag and re-deploy.

</details>

**Wallet check.** Still at $0.00. Two metadata deploys. The two jobs sit there idle; no compute spent.

## Cleanup

Time to tear it all down. The cell below deletes the deployed prod and dev jobs (the bundle's `bundle destroy --target ...` is the preferred way locally; the cleanup adapter does the API-side delete from the notebook). It then removes the workspace bundle artifact paths, drops the four tables, and drops the two schemas.

The local `labrun-bundle/` directory and the `~/.databrickscfg` profile are OUTSIDE this notebook's lifecycle. Delete them by hand if you do not want the scaffolding around:

```bash
rm -rf labrun-bundle/
# Optional: remove the CLI profile from ~/.databrickscfg
```

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST. Both jobs are
# critical (running deploys consume daily quota).

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_resources = [r for r in CLEANUP_MANIFEST if r.id not in failed_ids and getattr(r, "critical", False)]
standard_resources = [r for r in CLEANUP_MANIFEST if r.id not in failed_ids and not getattr(r, "critical", False)]
critical_destroyed = len(critical_resources)
standard_destroyed = len(standard_resources)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")
print()
print("Local cleanup reminder: rm -rf labrun-bundle/ if you no longer need the scaffolding.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.00.** Two metadata deploys, two pre-created schemas, two pre-created tables, and one bundle.yml authored locally. The expensive thing this session would have prevented is another prod overwrite incident shipping the dev catalog to a real customer dashboard.

## Reflection

These are not graded. They are for you.

1. Walk through what `databricks bundle deploy --target prod` does between the moment you press Enter and the moment the prod Lakeflow Job appears in the workspace UI. Name each step: YAML parse, variable resolution, artifact upload to the workspace, job-spec POST to the Jobs API, response handling. Where does the per-target variable substitution actually happen?

2. Your team currently has dev and prod in two separate Databricks workspaces (the production pattern Free Edition cannot demonstrate because Free Edition has one workspace). Sketch the bundle.yml differences vs the single-workspace lab pattern. What changes about authentication, target overrides, and the CI/CD pipeline that runs the bundle deploys?

## Exam-Style Practice

**Question 1 (Domain 5, bundle target variables):**

A Declarative Automation Bundle defines:

```yaml
variables:
  catalog_schema:
    default: workspace.default.labrun_bundle_dev
targets:
  prod:
    variables:
      catalog_schema: workspace.default.labrun_bundle_prod
```

A team member runs `databricks bundle deploy` (no --target flag) followed by `databricks bundle deploy --target prod`. What is the resolved value of `${var.catalog_schema}` in each deploy?

A. First deploy: `workspace.default.labrun_bundle_prod` (production is the default); second deploy: `workspace.default.labrun_bundle_prod`.
B. First deploy: `workspace.default.labrun_bundle_dev` (the default value); second deploy: `workspace.default.labrun_bundle_prod` (the prod target override).
C. First deploy: fails because no target was specified; second deploy: `workspace.default.labrun_bundle_prod`.
D. First deploy: an interactive prompt asking which target; second deploy: `workspace.default.labrun_bundle_prod`.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: bundle defaults are the default target's values, not the prod override's.
- B is correct: with no `--target` flag, the bundle CLI uses the default target (the first target defined, or the target marked `default: true`). The default uses the variable's default value. The prod target's override applies only when `--target prod` is passed.
- C is wrong: a no-target deploy is valid and uses defaults; it does not fail.
- D is wrong: the bundle CLI does not interactively prompt for target; it uses the default target.

</details>

**Question 2 (Domain 5, CLI authentication for CI/CD):**

A team wants their CI/CD pipeline to run `databricks bundle deploy --target prod` as part of an automated promotion workflow. The pipeline should NOT use a human's personal access token. Which authentication option is the recommended pattern in 2026?

A. Hardcode the team lead's PAT in the CI/CD secrets and rotate it monthly.
B. Use OAuth M2M (machine-to-machine) authentication with a service principal's client ID and client secret, configured as CI/CD environment variables.
C. Use the Databricks CLI's `auth login` command in the CI/CD job, which will open a browser for the team lead to approve every run.
D. Disable authentication and rely on workspace IP allow-listing to restrict who can deploy.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: a human PAT in CI/CD secrets is the anti-pattern the scenario explicitly rules out; the audit log attributes deploys to the human, not the pipeline.
- B is correct: OAuth M2M with a service principal is the documented Databricks CI/CD authentication pattern. The principal carries its own client ID and secret, the audit log attributes deploys to the principal, and the credentials rotate independently of any human.
- C is wrong: `auth login` opens a browser, which a CI/CD environment does not have. The OAuth user flow is for local dev, not CI/CD.
- D is wrong: Databricks does not support disabling authentication. Workspace IP allow-listing complements auth, it does not replace it.

</details>